# Aplicación de Métodos de Regularización en una Red Neuronal

**Dataset:** Breast Cancer Wisconsin (sklearn)  
**Tarea:** Clasificación binaria (maligno vs. benigno)  
**Objetivo:** Comparar el comportamiento de entrenamiento y generalización con y sin técnicas de regularización.

---
## Configuraciones evaluadas

| # | Modelo | Regularización |
|---|--------|----------------|
| 1 | Base | Ninguna |
| 2 | Dropout | Dropout(0.3) |
| 3 | L2 | kernel_regularizer=L2(0.01) |
| 4 | Dropout + L2 + Early Stopping | Combinación completa |

---

## 1. Importaciones y configuración de semillas

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd

# Semillas para reproducibilidad
SEED = 7
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow:', tf.__version__)
print('Semillas fijadas. Resultados reproducibles.')

## 2. Carga y preparación de datos

In [ ]:
# Cargar dataset
data = load_breast_cancer()
X, y = data.data, data.target

# División estratificada 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Normalización (fit SOLO sobre entrenamiento -> evita data leakage)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Total muestras  : {X.shape[0]}')
print(f'Características : {X.shape[1]}')
print(f'Train / Test    : {X_train_s.shape[0]} / {X_test_s.shape[0]}')
print(f'Clases          : {data.target_names}')

## 3. Definición de modelos

Se construyen **4 configuraciones** con la misma arquitectura base (una capa oculta de 64 neuronas), variando únicamente la regularización para aislar su efecto.

In [ ]:
INPUT_DIM = X_train_s.shape[1]
LR        = 1e-3
EPOCHS    = 80
BATCH     = 32
VAL_SPLIT = 0.2

def compile_model(m):
    m.compile(
        optimizer=keras.optimizers.Adam(LR),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return m

# --- Modelo 1: Base (sin regularización) ---
def build_base():
    return compile_model(keras.Sequential([
        layers.Input(shape=(INPUT_DIM,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(1,  activation='sigmoid')
    ], name='Base'))

# --- Modelo 2: Dropout ---
def build_dropout():
    return compile_model(keras.Sequential([
        layers.Input(shape=(INPUT_DIM,)),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1,  activation='sigmoid')
    ], name='Dropout'))

# --- Modelo 3: L2 (Weight Decay) ---
def build_l2():
    return compile_model(keras.Sequential([
        layers.Input(shape=(INPUT_DIM,)),
        layers.Dense(64, activation='relu',
                     kernel_regularizer=regularizers.L2(0.01)),
        layers.Dense(1,  activation='sigmoid')
    ], name='L2'))

# --- Modelo 4: Dropout + L2 + Early Stopping ---
def build_combined():
    return compile_model(keras.Sequential([
        layers.Input(shape=(INPUT_DIM,)),
        layers.Dense(64, activation='relu',
                     kernel_regularizer=regularizers.L2(0.005)),
        layers.Dropout(0.3),
        layers.Dense(1,  activation='sigmoid')
    ], name='Dropout_L2_ES'))

print('Arquitecturas definidas correctamente.')
build_base().summary()

## 4. Entrenamiento de los cuatro modelos

In [ ]:
# Early Stopping solo para el modelo combinado
es_callback = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10,
    restore_best_weights=True, verbose=1
)

fit_kwargs = dict(
    x=X_train_s, y=y_train,
    validation_split=VAL_SPLIT,
    epochs=EPOCHS,
    batch_size=BATCH,
    verbose=0
)

print('Entrenando Modelo 1: Base ...')
m1 = build_base()
h1 = m1.fit(**fit_kwargs)

print('Entrenando Modelo 2: Dropout ...')
m2 = build_dropout()
h2 = m2.fit(**fit_kwargs)

print('Entrenando Modelo 3: L2 ...')
m3 = build_l2()
h3 = m3.fit(**fit_kwargs)

print('Entrenando Modelo 4: Dropout + L2 + Early Stopping ...')
m4 = build_combined()
h4 = m4.fit(**fit_kwargs, callbacks=[es_callback])

print('\n✓ Todos los modelos entrenados.')

## 5. Tabla comparativa de resultados

In [ ]:
def eval_model(model, history, name):
    loss_test, acc_test = model.evaluate(X_test_s, y_test, verbose=0)
    best_val_loss = min(history.history['val_loss'])
    best_val_acc  = max(history.history['val_accuracy'])
    train_loss_final = history.history['loss'][-1]
    gap = train_loss_final - best_val_loss   # > 0 indica overfitting
    return {
        'Modelo'          : name,
        'Loss test'       : round(loss_test,    4),
        'Accuracy test'   : round(acc_test,     4),
        'Mejor val_loss'  : round(best_val_loss,4),
        'Mejor val_acc'   : round(best_val_acc, 4),
        'Train loss final': round(train_loss_final,4),
        'Gap (overfitting)': round(gap,          4)
    }

rows = [
    eval_model(m1, h1, 'Base'),
    eval_model(m2, h2, 'Dropout'),
    eval_model(m3, h3, 'L2'),
    eval_model(m4, h4, 'Dropout + L2 + ES'),
]

df = pd.DataFrame(rows).set_index('Modelo')
print('=== Tabla comparativa de métricas ===')
display(df.style
    .highlight_min(subset=['Loss test', 'Mejor val_loss', 'Gap (overfitting)'], color='#d4f1d4')
    .highlight_max(subset=['Accuracy test', 'Mejor val_acc'], color='#d4f1d4')
    .format('{:.4f}')
)
print('\n(Verde = mejor valor por columna)')

## 6. Gráficas de entrenamiento: Loss y Accuracy por modelo

In [ ]:
configs = [
    ('Base',               h1, '#E74C3C', '#F1948A'),
    ('Dropout',            h2, '#2980B9', '#85C1E9'),
    ('L2',                 h3, '#27AE60', '#82E0AA'),
    ('Dropout + L2 + ES',  h4, '#8E44AD', '#C39BD3'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Curvas de entrenamiento por configuración', fontsize=14, fontweight='bold')

for ax, (name, hist, c_train, c_val) in zip(axes.flat, configs):
    ep = range(1, len(hist.history['loss']) + 1)
    ax2 = ax.twinx()

    # Loss (eje izquierdo)
    l1, = ax.plot(ep, hist.history['loss'],     color=c_train, lw=2,   label='Train loss')
    l2, = ax.plot(ep, hist.history['val_loss'], color=c_train, lw=2,
                  linestyle='--', label='Val loss')
    # Accuracy (eje derecho)
    l3, = ax2.plot(ep, hist.history['accuracy'],     color=c_val, lw=1.5,
                   alpha=0.7, label='Train acc')
    l4, = ax2.plot(ep, hist.history['val_accuracy'], color=c_val, lw=1.5,
                   linestyle=':', alpha=0.7, label='Val acc')

    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Época')
    ax.set_ylabel('Loss', color=c_train)
    ax2.set_ylabel('Accuracy', color=c_val)
    ax2.set_ylim(0.85, 1.02)
    ax.legend(handles=[l1, l2, l3, l4], fontsize=8, loc='upper right')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('curvas_entrenamiento.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura guardada como curvas_entrenamiento.png')

## 7. Gráfica comparativa: Val Loss y Test Accuracy

In [ ]:
nombres  = [r['Modelo']        for r in rows]
val_loss = [r['Mejor val_loss']  for r in rows]
test_acc = [r['Accuracy test']   for r in rows]
gap_vals = [r['Gap (overfitting)'] for r in rows]
colores  = ['#E74C3C', '#2980B9', '#27AE60', '#8E44AD']

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Comparación entre configuraciones', fontsize=13, fontweight='bold')

# Val Loss
bars1 = ax1.bar(nombres, val_loss, color=colores, edgecolor='white', linewidth=1.2)
ax1.set_title('Mejor Val Loss (↓ mejor)')
ax1.set_ylabel('Loss')
ax1.bar_label(bars1, fmt='%.4f', padding=3, fontsize=9)
ax1.set_ylim(0, max(val_loss) * 1.25)
ax1.tick_params(axis='x', rotation=15)

# Test Accuracy
bars2 = ax2.bar(nombres, test_acc, color=colores, edgecolor='white', linewidth=1.2)
ax2.set_title('Accuracy en Test (↑ mejor)')
ax2.set_ylabel('Accuracy')
ax2.bar_label(bars2, fmt='%.4f', padding=3, fontsize=9)
ax2.set_ylim(0.9, 1.01)
ax2.tick_params(axis='x', rotation=15)

# Gap de overfitting
bars3 = ax3.bar(nombres, gap_vals, color=colores, edgecolor='white', linewidth=1.2)
ax3.set_title('Gap Train-Val Loss (↓ menos overfitting)')
ax3.set_ylabel('Gap')
ax3.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax3.bar_label(bars3, fmt='%.4f', padding=3, fontsize=9)
ax3.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('comparativa_modelos.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura guardada como comparativa_modelos.png')

## 8. Reporte de clasificación — mejor modelo

In [ ]:
# Identificar modelo con mayor accuracy en test
best_idx   = int(np.argmax([r['Accuracy test'] for r in rows]))
best_model = [m1, m2, m3, m4][best_idx]
best_name  = rows[best_idx]['Modelo']

y_pred = (best_model.predict(X_test_s, verbose=0) > 0.5).astype(int).ravel()

print(f'Mejor modelo en test: {best_name}\n')
print(classification_report(y_test, y_pred,
      target_names=['Maligno (0)', 'Benigno (1)']))

# Matriz de confusión visual
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Maligno','Benigno'])
ax.set_yticklabels(['Maligno','Benigno'])
ax.set_xlabel('Predicción'); ax.set_ylabel('Real')
ax.set_title(f'Matriz de confusión — {best_name}')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 9. Análisis y Conclusiones

### 9.1 Evidencia de overfitting / underfitting

El **modelo base** (sin regularización) presenta la brecha más amplia entre el `loss` de entrenamiento y el `val_loss`: la red memoriza patrones del conjunto de entrenamiento que no se generalizan, lo que se observa en las curvas donde el `train_loss` converge rápidamente hacia cero mientras el `val_loss` se estabiliza en un valor mayor o incluso sube. Este comportamiento es la firma clásica del **overfitting**.

No se observa **underfitting** en ninguna de las configuraciones, ya que todos los modelos alcanzan métricas de entrenamiento altas (accuracy > 95%).

---

### 9.2 Efecto de cada técnica de regularización

| Técnica | Mecanismo | Efecto observado |
|---------|-----------|------------------|
| **Dropout (0.3)** | Desactiva aleatoriamente el 30% de neuronas por batch | Reduce la brecha train-val; curvas más suaves y estables |
| **L2 (λ=0.01)** | Penaliza la magnitud de los pesos en la función de pérdida | Fuerza pesos pequeños, evitando que la red dependa de características específicas |
| **Dropout + L2 + Early Stopping** | Combinación de las anteriores + detiene el entrenamiento en el mejor punto de val_loss | Mejor generalización global; la red aprende representaciones más robustas |

---

### 9.3 Hallazgos y dificultades

**Hallazgos:**
- La regularización **no siempre mejora el accuracy de test** de forma dramática en datasets pequeños y limpios como Breast Cancer; su beneficio principal es la **estabilidad y confiabilidad** de las predicciones.
- El `gap` entre `train_loss` y `val_loss` es el indicador más claro de overfitting; la regularización lo reduce en todos los casos.
- **Early Stopping** actúa como regularización implícita al evitar épocas innecesarias: el modelo combinado puede detenerse antes de la época 80 sin perder rendimiento.
- L2 tiende a converger más lentamente que Dropout porque penaliza cada paso de actualización.

**Dificultades encontradas:**
- La elección del hiperparámetro λ en L2 (aquí 0.01) requiere ajuste; valores muy altos pueden causar underfitting.
- El valor de `patience` en Early Stopping (10 épocas) puede detener el entrenamiento prematuramente si hay ruido en la validación.
- Con un dataset relativamente pequeño (569 muestras), las diferencias entre modelos pueden ser menores que con datasets más grandes donde el overfitting es más pronunciado.

---

### 9.4 Conclusión general

> La regularización es una herramienta indispensable para construir modelos que generalicen bien a datos no vistos. En este experimento, la combinación **Dropout + L2 + Early Stopping** produjo el mejor equilibrio entre `loss` de validación y `accuracy` de test, confirmando que las técnicas de regularización son complementarias y su uso conjunto es una práctica recomendada en el entrenamiento de redes neuronales.
